# 01 — Data Extraction & Cohort Construction
Load MIMIC-IV discharge summaries from CSV files, join ICD-10 labels, and create patient-level splits.

## 1. Configuration

In [ ]:
import pandas as pd
import numpy as np
import os

# Local MIMIC-IV data directory 
LOCAL_DATA_DIR = '../datasets/raw'

# Output directory 
OUT_DIR = '../datasets/processed'
os.makedirs(OUT_DIR, exist_ok=True)

# Random seed
SEED = 42

# Train/val/test split 
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

MIN_CODE_FREQ = 10   # drop codes appearing in fewer than 10 notes

print(f'Local data dir: {LOCAL_DATA_DIR}')
print(f'Output dir:     {OUT_DIR}')

## 2. Load raw tables from local files

In [ ]:
# Load discharge notes, diagnoses, and procedures data from  CSV files into DataFrames

print('Loading discharge notes...')
notes = pd.read_csv(
    os.path.join(LOCAL_DATA_DIR, 'discharge.csv'),
    usecols=['subject_id', 'hadm_id', 'note_id', 'charttime', 'text']
)
print(f'  notes shape: {notes.shape}')

print('Loading diagnoses_icd...')
diag = pd.read_csv(
    os.path.join(LOCAL_DATA_DIR, 'diagnoses_icd.csv'),
    usecols=['subject_id', 'hadm_id', 'icd_code', 'icd_version']
)
print(f'  diagnoses shape: {diag.shape}')

print('Loading procedures_icd...')
proc = pd.read_csv(
    os.path.join(LOCAL_DATA_DIR, 'procedures_icd.csv'),
    usecols=['subject_id', 'hadm_id', 'icd_code', 'icd_version']
)
print(f'  procedures shape: {proc.shape}')

## 3. Filter to ICD-10 and combine

In [ ]:
# The data contains both ICD-9 and ICD-10 codes. we'll focus on ICD-10 codes only for this project and merge diagnoses and procedures together.
diag10 = diag[diag['icd_version'] == 10][['subject_id', 'hadm_id', 'icd_code']].copy()
proc10 = proc[proc['icd_version'] == 10][['subject_id', 'hadm_id', 'icd_code']].copy()


codes = pd.concat([diag10, proc10], ignore_index=True).drop_duplicates()
print(f'Total unique (hadm_id, icd_code) pairs: {len(codes)}')
print(f'Unique ICD-10 codes: {codes["icd_code"].nunique()}')

## 4. Keep one discharge note per admission

In [ ]:
# Some admissions have multiple discharge notes, we are using only the latest ones.
notes = notes.dropna(subset=['hadm_id', 'text'])
notes['hadm_id'] = notes['hadm_id'].astype(int)
notes = (notes
         .sort_values('charttime', ascending=False)
         .drop_duplicates(subset=['hadm_id'], keep='first')
         .reset_index(drop=True))
print(f'Discharge notes after dedup: {len(notes)}')

## 7. Join notes ↔ ICD-10 codes

In [ ]:
# Aggregate codes per admission → list
codes_agg = (codes
             .groupby('hadm_id')['icd_code']
             .apply(list)
             .reset_index()
             .rename(columns={'icd_code': 'icd_codes'}))


df = notes.merge(codes_agg, on='hadm_id', how='inner')
print(f'Cohort size (note + ICD-10 codes): {len(df)}')
df.head(2)

## 8. Build label vocabulary (frequency-filtered)

In [ ]:
# Count ICD codes and build vocabulary
# Keep only codes occuring more than 10 times in the dataset.

from collections import Counter

all_codes = [c for codes_list in df['icd_codes'] for c in codes_list]
code_freq = Counter(all_codes)


vocab = sorted([c for c, cnt in code_freq.items() if cnt >= MIN_CODE_FREQ])
vocab_set = set(vocab)
print(f'Full unique codes : {len(code_freq)}')
print(f'Vocab (freq>={MIN_CODE_FREQ}): {len(vocab)}')


pd.Series(vocab).to_csv(f'{OUT_DIR}/label_vocab.csv', index=False, header=['icd_code'])
print('Saved label_vocab.csv')

## 9. Patient-level train / val / test splits

In [ ]:
from sklearn.model_selection import train_test_split

patients = df['subject_id'].unique()
np.random.seed(SEED)

train_pat, temp_pat = train_test_split(patients, test_size=(VAL_FRAC + TEST_FRAC), random_state=SEED)
val_pat,   test_pat = train_test_split(temp_pat, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), random_state=SEED)

def assign_split(subject_id):
    if subject_id in set(train_pat): return 'train'
    if subject_id in set(val_pat):   return 'val'
    return 'test'

df['split'] = df['subject_id'].map(assign_split)
print(df['split'].value_counts())

## 10. Save processed dataset

In [ ]:
# Store icd_codes list as pipe-separated string for easy CSV storage
df['icd_codes_str'] = df['icd_codes'].apply(lambda x: '|'.join(x))

cols_out = ['subject_id', 'hadm_id', 'note_id', 'text', 'icd_codes_str', 'split']
df[cols_out].to_parquet(f'{OUT_DIR}/cohort_full.parquet', index=False)

for split in ['train', 'val', 'test']:
    subset = df[df['split'] == split][cols_out]
    subset.to_parquet(f'{OUT_DIR}/cohort_{split}.parquet', index=False)
    print(f'  {split}: {len(subset)} rows')

print('All splits saved to', OUT_DIR)

## 11. Label & note-length distributions plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

train_df = df[df['split'] == 'train'].copy()

# Note length - word count distribution plot
train_df['word_count'] = train_df['text'].str.split().apply(len)
plt.figure(figsize=(8, 3))
train_df['word_count'].clip(upper=4000).hist(bins=80, color='steelblue')
plt.xlabel('Word count'); plt.ylabel('# notes'); plt.title('Discharge note length (train)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/note_length_dist.png', dpi=120); plt.show()

# Label cardinality plot
train_df['n_codes'] = train_df['icd_codes'].apply(len)
plt.figure(figsize=(8, 3))
train_df['n_codes'].clip(upper=50).hist(bins=50, color='darkorange')
plt.xlabel('# ICD codes per note'); plt.ylabel('# notes'); plt.title('Label cardinality (train)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/label_cardinality.png', dpi=120); plt.show()

# top-200 code frequencies plot
freq_series = pd.Series(code_freq).sort_values(ascending=False)
plt.figure(figsize=(10, 3))
freq_series.iloc[:200].plot(logy=True, color='green')
plt.xlabel('Code rank'); plt.ylabel('Frequency (log)'); plt.title('ICD-10 code frequency (top 200)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/code_freq_tail.png', dpi=120); plt.show()